# Laya V4.8 — SVD Delta-Shared Local Attention (Fidelity-First)

[Open in Google Colab](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V48_SVD_Delta_Shared_Attention_Colab.ipynb)

This new experiment addresses V4.7's **85% teacher agreement and 10.83 percentage-point gold-accuracy loss** on its best 120-decision probe.

**New model:** retain the pretrained ModernBERT 28-layer FULL → LOCAL(128) → LOCAL(128) encoder and the successful V4.6 64-token marker-aware decision head, but share the first local layer's attention projections with the next local layer plus a learned low-rank residual:

`W_second = W_first + A @ B`

The matrices A and B are initialized using truncated SVD of the difference between the **original pretrained** second and first layers. The two FFNs, layer norms, RoPE and residuals remain separate. All global layers are unchanged.

**Workflow:** load [V4.6 standalone](https://huggingface.co/vtava/Laya-1.2-speedup-standalone), test ranks 32/64/128/256 on a case-disjoint PROBE, distill only A/B matrices, select a PROBE checkpoint, run one untouched AUDIT, benchmark latency vs V4.6, export a full offline FP32 standalone model, check reload parity, and publish only if accuracy, size and speed gates all pass.

**Target (conditional):** `vtava/Laya-v48-svd-delta-shared` (private by default). The existing model repositories are left untouched. Weight reduction **does not guarantee lower GPU latency**: the added low-rank matrix operations may slow inference. No results are claimed until you run this notebook on Colab GPU.


In [ ]:
#@title 1. Dependencies, GPU and imports
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
import json,math,random,time,gc,importlib.util,copy,statistics,hashlib
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from datasets import load_dataset
from huggingface_hub import HfApi, snapshot_download, notebook_login
from safetensors.torch import save_file, load_file
from transformers import AutoTokenizer
assert torch.cuda.is_available(), "Select GPU in Colab runtime settings"
print("GPU:",torch.cuda.get_device_name(0),"torch:",torch.__version__)


In [ ]:
#@title 2. Config and download complete V4.6 reference model
TEACHER_REPO="vtava/Laya-1.2-speedup-standalone"
STUDENT_REPO="vtava/Laya-v48-svd-delta-shared"
SEED=2047 # same train/probe/audit split seed as V4.7
MAX_LEN=512 #@param {type:"integer"}
TRAIN_CASES=500
DEV_CASES=240
BATCH=2 #@param {type:"integer"}
TRAIN_B=1 #@param {type:"integer"}
TRAIN_MAX_TOKENS=224 #@param {type:"integer"}
TRAIN_STEPS=400 #@param {type:"integer"}
TRAIN_CHECK_EVERY=80 #@param {type:"integer"}
TRAIN_LR=0.00002 #@param {type:"number"}
CANDIDATE_RANKS=(32,64,128,256)
PROBE_DECISIONS=120
AUDIT_DECISIONS=200
BENCH_REPEATS=10 #@param {type:"integer"}
MIN_TEACHER_AGREEMENT=0.99
MAX_GOLD_DROP_PP=1.5
MAX_JS=0.003
MIN_GPU_SPEED_RATIO=0.95
MIN_WALL_SPEED_RATIO=0.95
MIN_SAVED_BYTES=100_000_000
PUBLISH_TO_HF=True #@param {type:"boolean"}
PRIVATE_REPO=True #@param {type:"boolean"}
ALLOW_UPDATE_EXISTING=False #@param {type:"boolean"}
assert 1<=TRAIN_B<=2 and 1<=BATCH<=8 and TRAIN_STEPS>=0
assert TRAIN_CHECK_EVERY>0 and TRAIN_MAX_TOKENS>=192 and BENCH_REPEATS>=5
device=torch.device("cuda")
amp_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False)
           else torch.float16)
OUT=Path("/content/Laya_V48_SVD_Delta_Shared")
OUT.mkdir(exist_ok=True,parents=True)
DEST=OUT/"standalone_model"
DEST.mkdir(exist_ok=True,parents=True)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
api=HfApi()
try: teacher_info=api.model_info(TEACHER_REPO)
except Exception as e:
    print("Private model may require Hugging Face authorization:",repr(e))
    notebook_login()
    teacher_info=api.model_info(TEACHER_REPO)
parent_dir=Path(snapshot_download(TEACHER_REPO,revision=teacher_info.sha))
source_config=json.loads((parent_dir/"config.json").read_text())
assert source_config["format"]=="laya-1p2-full-standalone-v1"
assert int(source_config["head_context_budget"])==64
source_cfg={"head_max_len":192,"max_len":MAX_LEN}
tokenizer=AutoTokenizer.from_pretrained(str(parent_dir/"tokenizer"),local_files_only=True)
assert tokenizer.pad_token_id is not None
print("Reference:",TEACHER_REPO,"@",teacher_info.sha)
print("Reference FP32 checkpoint:",(parent_dir/"model.safetensors").stat().st_size,"bytes")


In [ ]:
#@title 3. Original encoder and SVD-initialized low-rank local-layer deltas
# Embedded source is also published in TinyCeNN-LM standalone/laya_v48_svd_delta.
V48_RUNTIME_CODE = "\"\"\"Laya V4.8 SVD-initialized low-rank delta attention, fully standalone.\n\nRequires only torch, transformers, safetensors. Does NOT download or import\nconvaiinnovations/laya-typed-decisions or TinyCeNN-LM at inference.\nThe model weights and tokenizer must be present in the same export directory.\nOption-score model only; auxiliary action logits are intentionally unavailable.\n\"\"\"\nfrom __future__ import annotations\nimport json\nfrom pathlib import Path\nfrom typing import Any, Mapping, Sequence\nimport torch\nfrom torch import Tensor, nn\nfrom torch.nn import functional as F\nfrom safetensors.torch import load_file\nfrom transformers import AutoModel, AutoConfig, AutoTokenizer\n\nQTYPES = {\"choice\": 0, \"score\": 1, \"noul\": 2}\n_DEFAULT_NOUL_LABELS = {\"false\": \"false\", \"true\": \"true\"}\n\n\ndef _serialize_state(state: str | dict | list) -> str:\n    if isinstance(state, str):\n        return state\n    return json.dumps(state, ensure_ascii=False, separators=(\",\", \":\"), default=str)\n\n\ndef _render_value(value: Any) -> str:\n    if isinstance(value, str):\n        return value\n    return json.dumps(value, ensure_ascii=False, separators=(\", \", \": \"), default=str)\n\n\ndef normalize_question(question: Mapping[str, Any]) -> dict[str, Any]:\n    \"\"\"Normalize an external typed-decision question into the runtime format.\"\"\"\n    qtype = str(question.get(\"type\", question.get(\"t\", \"choice\"))).lower()\n    if qtype not in QTYPES:\n        raise ValueError(f\"unsupported question type: {qtype!r}\")\n\n    criteria = question.get(\"criteria\", question.get(\"crit\"))\n    if qtype == \"choice\":\n        if isinstance(criteria, (list, tuple)):\n            criteria = {str(x): None for x in criteria}\n        if not isinstance(criteria, Mapping) or len(criteria) < 2:\n            raise ValueError(\"choice questions require at least two criteria/options\")\n        criteria = dict(criteria)\n    elif qtype == \"score\":\n        if not isinstance(criteria, (list, tuple)) or len(criteria) < 2:\n            raise ValueError(\"score questions require at least two ordered criteria\")\n        criteria = list(criteria)\n    else:\n        if criteria is None:\n            criteria = {}\n        if not isinstance(criteria, Mapping):\n            raise ValueError(\"noul criteria must be a mapping when provided\")\n        criteria = {str(k).lower(): v for k, v in criteria.items()}\n\n    instructions = question.get(\"instructions\", question.get(\"ins\", \"\"))\n    if not isinstance(instructions, str):\n        instructions = json.dumps(instructions, ensure_ascii=False, default=str)\n\n    result = {\"t\": qtype, \"ins\": instructions, \"crit\": criteria}\n    if \"labels\" in question:\n        result[\"labels\"] = dict(question[\"labels\"])\n    return result\n\n\ndef render_options(question: Mapping[str, Any]) -> list[str]:\n    q = normalize_question(question) if \"t\" not in question else dict(question)\n    qtype, criteria = q[\"t\"], q.get(\"crit\")\n    if qtype == \"choice\":\n        return [\n            str(k) if v is None or v == \"\" else f\"{k}: {_render_value(v)}\"\n            for k, v in criteria.items()\n        ]\n    if qtype == \"score\":\n        return [f\"level {i}: {_render_value(v)}\" for i, v in enumerate(criteria)]\n\n    labels = q.get(\"labels\", _DEFAULT_NOUL_LABELS)\n    false_label = str(labels.get(\"false\", \"false\")).strip() or \"false\"\n    true_label = str(labels.get(\"true\", \"true\")).strip() or \"true\"\n    false_desc = criteria.get(\"false\")\n    true_desc = criteria.get(\"true\")\n    return [\n        f\"{false_label}: \" + (\n            _render_value(false_desc) if false_desc not in (None, \"\") else \"no, the statement does not hold\"\n        ),\n        f\"{true_label}: \" + (\n            _render_value(true_desc) if true_desc not in (None, \"\") else \"yes, the statement holds\"\n        ),\n    ]\n\n\ndef build_sequence(\n    tokenizer,\n    state: str | dict | list,\n    question: Mapping[str, Any],\n    *,\n    max_len: int = 512,\n    head_max_len: int = 192,\n    truncate_left: bool = False,\n    state_ids: Sequence[int] | None = None,\n) -> tuple[list[int], list[int]]:\n    \"\"\"Build a compact typed-decision sequence and option marker positions.\"\"\"\n    q = normalize_question(question) if \"t\" not in question else dict(question)\n    options = render_options(q)\n    mask_token = tokenizer.mask_token\n    instruction = str(q[\"ins\"]).replace(mask_token, \" \")\n    head_ids = tokenizer(\n        f\"{q['t']} question: {instruction}\", add_special_tokens=False\n    )[\"input_ids\"]\n\n    option_ids: list[list[int]] = []\n    for option in options:\n        ids = tokenizer(\n            \" \" + option.replace(mask_token, \" \"),\n            add_special_tokens=False,\n            truncation=True,\n            max_length=48,\n        )[\"input_ids\"]\n        option_ids.append([tokenizer.mask_token_id] + ids)\n\n    option_budget = head_max_len - sum(len(x) for x in option_ids)\n    if option_budget < 16:\n        per = max(4, (head_max_len - 16) // max(1, len(option_ids)))\n        option_ids = [x[:per] for x in option_ids]\n        option_budget = head_max_len - sum(len(x) for x in option_ids)\n    head_ids = head_ids[: max(8, option_budget)]\n\n    ids = [tokenizer.cls_token_id] + head_ids + [tokenizer.sep_token_id]\n    markers: list[int] = []\n    for option in option_ids:\n        markers.append(len(ids))\n        ids.extend(option)\n    ids.append(tokenizer.sep_token_id)\n\n    if state_ids is None:\n        text = _serialize_state(state).replace(mask_token, \" \")\n        state_ids = tokenizer(text, add_special_tokens=False)[\"input_ids\"]\n    room = max(0, max_len - len(ids) - 1)\n    if truncate_left:\n        start = max(0, len(state_ids) - room)\n        state_part = list(state_ids[start:])\n    else:\n        state_part = list(state_ids[:room])\n    ids = (ids + state_part + [tokenizer.sep_token_id])[:max_len]\n    markers = [m for m in markers if m < max_len]\n    return ids, markers\n\n\ndef collate_items(items: Sequence[Mapping[str, Any]], pad_id: int) -> dict[str, Tensor]:\n    if not items:\n        raise ValueError(\"items must not be empty\")\n    n = len(items)\n    length = max(len(x[\"ids\"]) for x in items)\n    kmax = max(len(x[\"markers\"]) for x in items)\n    ids = torch.full((n, length), int(pad_id), dtype=torch.long)\n    mask = torch.zeros((n, length), dtype=torch.long)\n    marker_pos = torch.zeros((n, kmax), dtype=torch.long)\n    marker_mask = torch.zeros((n, kmax), dtype=torch.bool)\n    qtype = torch.zeros(n, dtype=torch.long)\n\n    for i, item in enumerate(items):\n        row = torch.tensor(item[\"ids\"], dtype=torch.long)\n        ids[i, : row.numel()] = row\n        mask[i, : row.numel()] = 1\n        m = torch.tensor(item[\"markers\"], dtype=torch.long)\n        marker_pos[i, : m.numel()] = m\n        marker_mask[i, : m.numel()] = True\n        qtype[i] = int(item[\"qtype\"])\n    return {\n        \"input_ids\": ids,\n        \"attention_mask\": mask,\n        \"marker_pos\": marker_pos,\n        \"marker_mask\": marker_mask,\n        \"qtype\": qtype,\n    }\n\n\n\ndef _apply_rope_compat(cfg):\n    rope = getattr(cfg, \"rope_parameters\", None)\n    if not isinstance(rope,dict):\n        return\n    flat = rope.get(\"rope_theta\")\n    for layer_type, attr in ((\"full_attention\",\"global_rope_theta\"),\n                             (\"sliding_attention\",\"local_rope_theta\")):\n        v = rope.get(layer_type)\n        theta = v.get(\"rope_theta\") if isinstance(v,dict) else flat\n        if theta is not None and hasattr(cfg,attr):\n            setattr(cfg,attr,float(theta))\n\n\nclass LayaStandalone(nn.Module):\n    \"\"\"Native pretrained ModernBERT + exact 64-token option-scoring head.\n\n    ALL weights are stored locally in model.safetensors. No auxiliary act_head.\n    \"\"\"\n    def __init__(self, encoder_config=None, *, budget=64, head_layers=2,\n                 pretrained_components=None):\n        super().__init__()\n        self.budget = int(budget)\n        assert self.budget == 64, \"This checkpoint implements the validated 64-token head\"\n        if pretrained_components is not None:\n            # EXPORT ONLY: reuse the loaded upstream model without duplicating\n            # ~400M weights; this branch is not needed during standalone load.\n            self.encoder = pretrained_components.encoder\n            self.head = pretrained_components.head\n            self.type_emb = pretrained_components.type_emb\n            self.scorer = pretrained_components.scorer\n        else:\n            assert encoder_config is not None\n            _apply_rope_compat(encoder_config)\n            self.encoder = AutoModel.from_config(\n                encoder_config, attn_implementation=\"sdpa\")\n            d = int(encoder_config.hidden_size)\n            nhead = max(1,d//64)\n            layer = nn.TransformerEncoderLayer(\n                d,nhead,dim_feedforward=4*d,dropout=0.1,\n                batch_first=True,norm_first=True)\n            self.head = nn.TransformerEncoder(\n                layer,int(head_layers),enable_nested_tensor=False)\n            self.type_emb = nn.Embedding(3,d)\n            self.scorer = nn.Sequential(nn.LayerNorm(d),nn.Linear(d,d),\n                                        nn.GELU(),nn.Linear(d,1))\n\n    def _select_head_tokens(self,h,attention_mask,marker_pos,marker_mask):\n        # Copied from V4.6, keeping exact order/masks for reproduction.\n        b,t,d=h.shape\n        cls=torch.zeros((b,1),dtype=torch.long,device=h.device)\n        lm_count=min(self.budget,max(t-1,1))\n        lm=torch.linspace(1,max(t-1,1),steps=lm_count,device=h.device).round().long()\n        lm=lm.clamp(max=t-1).view(1,-1).expand(b,-1)\n        marker_ok=marker_mask.bool()\n        lm_ok=attention_mask.bool().gather(1,lm)\n        lm_ok=lm_ok & (lm!=0)\n        duplicate=(lm[:,:,None]==marker_pos[:,None,:]) & marker_ok[:,None,:]\n        lm_ok=lm_ok & ~duplicate.any(-1)\n        idx=torch.cat((cls,marker_pos.clamp(0,t-1),lm),dim=1)\n        keep=torch.cat((torch.ones((b,1),dtype=torch.bool,device=h.device),\n                        marker_ok,lm_ok),dim=1)\n        gathered=h.gather(1,idx[:,:,None].expand(-1,-1,d))\n        return gathered,keep\n\n    def forward(self,input_ids,attention_mask,marker_pos,marker_mask,qtype):\n        h=self.encoder(input_ids=input_ids,\n                       attention_mask=attention_mask).last_hidden_state\n        h=h+self.type_emb(qtype)[:,None,:]\n        compact,keep=self._select_head_tokens(\n            h,attention_mask,marker_pos,marker_mask)\n        transformed=self.head(compact,src_key_padding_mask=~keep)\n        marker_hidden=transformed[:,1:1+marker_pos.shape[1],:]\n        logits=self.scorer(marker_hidden).squeeze(-1).float()\n        logits=logits.masked_fill(~marker_mask,-1e4)\n        return logits,None\n\n    @torch.inference_mode()\n    def predict(self,tokenizer,state,questions,*,max_len=512,head_max_len=192):\n        items=[];names=[];labels=[]\n        for name,raw in questions.items():\n            q=normalize_question(raw)\n            if q[\"t\"]!=\"choice\":\n                raise ValueError(\"Only choice questions supported by predict()\")\n            opts=list(q[\"crit\"].keys())\n            ids,markers=build_sequence(tokenizer,state,q,max_len=max_len,\n                                       head_max_len=head_max_len)\n            if len(markers)!=len(opts):\n                raise ValueError(\"Truncated an option marker\")\n            names.append(name);labels.append(opts)\n            items.append(dict(ids=ids,markers=markers,qtype=QTYPES[q[\"t\"]]))\n        batch=collate_items(items,int(tokenizer.pad_token_id))\n        device=next(self.parameters()).device\n        batch={k:v.to(device) for k,v in batch.items()}\n        with (torch.autocast(\"cuda\",dtype=torch.float16)\n              if device.type==\"cuda\" else __import__(\"contextlib\").nullcontext()):\n            logits,_=self(**batch)\n        probs=logits.float().masked_fill(~batch[\"marker_mask\"],-1e9).softmax(-1).cpu()\n        answers={}\n        for i,name in enumerate(names):\n            p=probs[i,:len(labels[i])]\n            best=int(p.argmax())\n            answers[name]=dict(choice=labels[i][best],\n                               confidence=float(p[best]),\n                               probabilities={label:float(pr) for label,pr in zip(labels[i],p)})\n        return answers\n\n\ndef load_model(model_dir,*,device=\"cpu\",local_files_only=True,compile_head=False):\n    \"\"\"Load entirely from the local standalone directory. No network needed.\"\"\"\n    root=Path(model_dir).resolve()\n    metadata=json.loads((root/\"config.json\").read_text())\n    assert metadata[\"format\"]==\"laya-1p2-full-standalone-v1\"\n    enc_cfg=AutoConfig.from_pretrained(\n        str(root/\"encoder\"),local_files_only=local_files_only)\n    model=LayaStandalone(enc_cfg,budget=int(metadata[\"head_context_budget\"]),\n                         head_layers=int(metadata[\"head_layers\"]))\n    state=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(state,strict=True)\n    model=model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tok=AutoTokenizer.from_pretrained(\n        str(root/\"tokenizer\"),local_files_only=local_files_only)\n    return model,tok\n\ndef local_pair_indices(model):\n    \"\"\"Verify FULL / same-window LOCAL / LOCAL triples in ModernBERT.\n\n    transformers==4.57.6 uses attn.local_attention: (-1,-1) means FULL,\n    (64,64) is a 128-token sliding window. Newer builds can expose\n    config.layer_types with attn.sliding_window instead. Support both.\n    \"\"\"\n    layers=model.encoder.layers\n    cfg=model.encoder.config\n    n=len(layers)\n    layer_types=getattr(cfg,\"layer_types\",None)\n    if layer_types is not None and len(layer_types)!=n:\n        raise ValueError(\"ModernBERT layer_types count does not match layers\")\n\n    def mode_at(i):\n        a=layers[i].attn\n        old_window=getattr(a,\"local_attention\",None)\n        if old_window is not None:\n            window=tuple(old_window)\n            if window==(-1,-1):\n                return (\"full\",None)\n            if len(window)!=2 or min(window)<0:\n                raise ValueError(f\"Malformed local attention window at {i}: {window}\")\n            return (\"local\",window)\n        typ=(layer_types[i] if layer_types is not None else\n             getattr(layers[i],\"attention_type\",None))\n        window=getattr(a,\"sliding_window\",None)\n        if typ==\"full_attention\" and window is None:\n            return (\"full\",None)\n        if typ==\"sliding_attention\" and window is not None:\n            return (\"local\",int(window))\n        raise ValueError(f\"Unsupported ModernBERT attention metadata at layer {i}: {typ}, {window}\")\n\n    pairs=[]\n    for anchor in range(0,n,3):\n        mode,_=mode_at(anchor)\n        if mode!=\"full\":\n            raise ValueError(f\"Layer {anchor} must be FULL attention\")\n        left_idx,right_idx=anchor+1,anchor+2\n        if right_idx>=n:\n            continue\n        m1,w1=mode_at(left_idx)\n        m2,w2=mode_at(right_idx)\n        if m1!=\"local\" or m2!=\"local\" or w1!=w2:\n            raise ValueError(\n                f\"Layers {(left_idx,right_idx)} must have identical local windows: {m1}/{w1}, {m2}/{w2}\")\n        pairs.append((left_idx,right_idx))\n    return pairs\n\n\nclass SVDDeltaLinear(nn.Module):\n    \"\"\"Second-layer linear projection = shared first-layer projection + A @ B.\n\n    The shared linear is a plain Python reference, NOT a registered submodule;\n    its weights appear only once in state_dict, under the original first layer.\n    \"\"\"\n    def __init__(self, first_linear, rank, *, A=None, B=None):\n        super().__init__()\n        if not isinstance(first_linear,nn.Linear):\n            raise TypeError(f\"Expected nn.Linear, got {type(first_linear)}\")\n        out_dim,in_dim=first_linear.weight.shape\n        rank=int(rank)\n        if not (1<=rank<min(in_dim,out_dim)):\n            raise ValueError(f\"Unsupported rank {rank} for {out_dim}x{in_dim}\")\n        # Using object.__setattr__ prevents double registering the shared module.\n        object.__setattr__(self,\"_shared\",first_linear)\n        self.in_features=int(in_dim)\n        self.out_features=int(out_dim)\n        self.max_rank=rank\n        self.active_rank=rank\n        self.A=nn.Parameter(torch.zeros((out_dim,rank),device=first_linear.weight.device,\n                                        dtype=first_linear.weight.dtype))\n        self.B=nn.Parameter(torch.zeros((rank,in_dim),device=first_linear.weight.device,\n                                        dtype=first_linear.weight.dtype))\n        if A is not None and B is not None:\n            with torch.no_grad():\n                self.A.copy_(A.to(self.A.device,self.A.dtype))\n                self.B.copy_(B.to(self.B.device,self.B.dtype))\n\n    def set_rank(self, rank):\n        rank=int(rank)\n        if rank<1 or rank>self.max_rank:\n            raise ValueError(f\"rank {rank} outside 1..{self.max_rank}\")\n        self.active_rank=rank\n\n    def shrink(self, rank):\n        \"\"\"Permanently discard extra SVD columns for a compact checkpoint.\"\"\"\n        rank=int(rank)\n        if rank<1 or rank>self.max_rank:\n            raise ValueError(rank)\n        self.A=nn.Parameter(self.A[:,:rank].detach().clone())\n        self.B=nn.Parameter(self.B[:rank,:].detach().clone())\n        self.max_rank=rank\n        self.active_rank=rank\n\n    def forward(self,x):\n        # fp16 autocast handles both paths identically, with FP32 stored weights.\n        w=self._shared.weight\n        bias=self._shared.bias\n        base=F.linear(x,w,bias)\n        r=self.active_rank\n        return base+F.linear(F.linear(x,self.B[:r,:]),self.A[:,:r])\n\n\ndef initialize_svd_delta(model, *, max_rank=256, oversampling=12, niter=2,\n                         initialized=True):\n    \"\"\"Tie 9 local pairs and keep a truncated SVD of each follower-minus-leader.\n\n    On export reload, initialized=False creates zero adapters and avoids\n    recalculating SVD; checkpoint load overwrites A and B.\n    \"\"\"\n    pairs=local_pair_indices(model)\n    info=[]\n    for first,second in pairs:\n        attention_1=model.encoder.layers[first].attn\n        attention_2=model.encoder.layers[second].attn\n        for name in (\"Wqkv\",\"Wo\"):\n            leader=getattr(attention_1,name)\n            follower=getattr(attention_2,name)\n            if not isinstance(leader,nn.Linear) or not isinstance(follower,nn.Linear):\n                raise TypeError(f\"{name} must be nn.Linear\")\n            if leader.weight.shape!=follower.weight.shape:\n                raise ValueError(f\"{name} unequal weight shapes\")\n            if leader.bias is not None or follower.bias is not None:\n                # Bias can be added to SVD basis via augmented feature, but\n                # this model is bias-free; assert to avoid silent loss of bias.\n                raise ValueError(f\"Biasful {name} unsupported; investigate first\")\n            rank=min(int(max_rank),min(leader.weight.shape)-1)\n            A=B=None\n            relative_error=None\n            if initialized:\n                with torch.no_grad():\n                    delta=(follower.weight.float()-leader.weight.float())\n                    q=min(rank+int(oversampling),min(delta.shape))\n                    U,S,V=torch.svd_lowrank(delta,q=q,niter=int(niter))\n                    root=S[:rank].clamp_min(0).sqrt()\n                    A=U[:,:rank]*root[None,:]\n                    B=root[:,None]*V[:,:rank].T\n                    captured=((S[:rank]**2).sum()/\n                             delta.square().sum().clamp_min(1e-20)).clamp(0,1)\n                    relative_error=float((1-captured).sqrt().item())\n                    del delta,U,S,V\n            setattr(attention_2,name,SVDDeltaLinear(leader,rank,A=A,B=B))\n            info.append(dict(first=int(first),second=int(second),component=name,\n                             shape=list(leader.weight.shape),max_rank=rank,\n                             svd_relative_frobenius_error=relative_error))\n    return pairs,info\n\n\ndef adapter_modules(model):\n    out=[]\n    for first,second in local_pair_indices(model):\n        for name in (\"Wqkv\",\"Wo\"):\n            adapter=getattr(model.encoder.layers[second].attn,name)\n            if not isinstance(adapter,SVDDeltaLinear):\n                raise AssertionError(f\"Missing SVD adapter at layer {second}.{name}\")\n            out.append(adapter)\n    return out\n\n\ndef change_active_rank(model,rank):\n    for adapter in adapter_modules(model):\n        adapter.set_rank(rank)\n\n\ndef shrink_all_adapters(model,rank):\n    for adapter in adapter_modules(model):\n        adapter.shrink(rank)\n\n\ndef unique_parameter_count(model):\n    return sum(p.numel() for p in model.parameters())\n\n\ndef load_v48_model(model_dir, *, device=\"cpu\", compile_head=False):\n    \"\"\"Reload compressed checkpoint completely offline, from ONE directory.\"\"\"\n    root=Path(model_dir).resolve()\n    meta=json.loads((root/\"config.json\").read_text())\n    if meta[\"format\"]!=\"laya-v48-svd-delta-v1\":\n        raise ValueError(\"Not a V4.8 SVD-delta model\")\n    cfg=AutoConfig.from_pretrained(str(root/\"encoder\"),local_files_only=True)\n    model=LayaStandalone(cfg,budget=int(meta[\"head_context_budget\"]),\n                         head_layers=int(meta[\"head_layers\"]))\n    pairs,info=initialize_svd_delta(\n        model,max_rank=int(meta[\"rank\"]),initialized=False)\n    if [list(p) for p in pairs]!=meta[\"shared_attention_pairs\"]:\n        raise ValueError(\"Pair pattern mismatch\")\n    weights=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(weights,strict=True)\n    for first,second in pairs:\n        for name in (\"Wqkv\",\"Wo\"):\n            w=getattr(model.encoder.layers[second].attn,name)\n            assert isinstance(w,SVDDeltaLinear)\n            assert \"_shared\" not in w._modules\n            assert w._shared is getattr(model.encoder.layers[first].attn,name)\n    model=model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tokenizer=AutoTokenizer.from_pretrained(\n        str(root/\"tokenizer\"),local_files_only=True)\n    return model,tokenizer\n"
runtime_file=OUT/"modeling_laya_v48.py"
runtime_file.write_text(V48_RUNTIME_CODE,encoding="utf-8")
spec=importlib.util.spec_from_file_location("v48_standalone",str(runtime_file))
assert spec and spec.loader
v48=importlib.util.module_from_spec(spec)
spec.loader.exec_module(v48)
# These definitions are REQUIRED by the V4.6 case-disjoint dataset cell below.
normalize_question=v48.normalize_question
build_sequence=v48.build_sequence
collate_items=v48.collate_items
QTYPES=v48.QTYPES
assert all(callable(f) for f in (normalize_question,build_sequence,collate_items))
teacher,ttok=v48.load_model(parent_dir,device="cuda")
student,stok=v48.load_model(parent_dir,device="cuda")
teacher.eval().requires_grad_(False)
student.eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
student.encoder.config.reference_compile=False
native_unique=v48.unique_parameter_count(teacher)
pairs,svd_diagnostics=v48.initialize_svd_delta(
    student,max_rank=max(CANDIDATE_RANKS),oversampling=12,niter=2)
assert len(pairs)==9,pairs
assert len(v48.adapter_modules(student))==18
print("Paired LOCAL layers:",pairs)
print("Total unique V4.6 parameters:",f"{native_unique:,}")
print("Original-delta SVD diagnostics:",svd_diagnostics)


In [ ]:
#@title 4. CASE-DISJOINT TRAIN / PROBE / AUDIT benchmark from V4.6
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
# Split at the CASE level before creating multiple question items per case.
probe_rows = dev_rows[:len(dev_rows) // 2]
audit_rows = dev_rows[len(dev_rows) // 2:]
probe_items = make_items_from_rows(probe_rows, SEED + 1, True)
audit_items = make_items_from_rows(audit_rows, SEED + 2, True)
dev_items = probe_items # backward-compatible alias for training helper use
assert train_items and probe_items and audit_items
assert {str(r["id"]) for r in probe_rows}.isdisjoint({str(r["id"]) for r in audit_rows})

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

assert len(probe_items)>=PROBE_DECISIONS and len(audit_items)>=AUDIT_DECISIONS
assert {x["case_id"] for x in probe_items}.isdisjoint({x["case_id"] for x in audit_items})
print("TRAIN/PROBE/AUDIT items:",len(train_items),len(probe_items),len(audit_items))


In [ ]:
#@title 5. Accuracy, divergence and paired CUDA/wall timing
@torch.inference_mode()
def probs_for(model,b):
    with torch.autocast("cuda",dtype=amp_dtype):
        scores,_=model(**b)
    return torch.softmax(scores.float().masked_fill(
        ~b["marker_mask"].bool(),-1e9),dim=-1).cpu()

@torch.inference_mode()
def get_reference(items,limit):
    out=[]
    for start in range(0,min(limit,len(items)),BATCH):
        chosen=items[start:min(start+BATCH,limit)]
        p=probs_for(teacher,direct_batch(chosen))
        for j,item in enumerate(chosen):
            out.append(dict(probs=p[j],teacher=int(p[j].argmax()),
                gold=int(item["gold_index"]),case=item["case_id"],qid=item["qid"]))
    return out

@torch.inference_mode()
def quality(model,items,ref):
    agree=correct=tcorrect=0;js=0.
    n=len(ref)
    for start in range(0,n,BATCH):
        chosen=items[start:min(n,start+BATCH)]
        p=probs_for(model,direct_batch(chosen))
        for j,item in enumerate(chosen):
            t=ref[start+j]["probs"]; s=p[j]
            assert s.shape==t.shape
            m=(s+t)/2
            js+=float(.5*((s*(s.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum()+
                            (t*(t.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum()))
            guess=int(s.argmax())
            agree+=int(guess==ref[start+j]["teacher"])
            correct+=int(guess==item["gold_index"])
            tcorrect+=int(ref[start+j]["teacher"]==item["gold_index"])
    return dict(decisions=n,teacher_agreement=agree/n,gold_accuracy=correct/n,
        teacher_gold_accuracy=tcorrect/n,gold_drop=(tcorrect-correct)/n,mean_js=js/n)

BENCH_BATCHES=[batch(probe_items,0,BATCH),batch(probe_items,11,BATCH)]
@torch.inference_mode()
def benchmark_paired(reference,candidate,repeats=BENCH_REPEATS):
    """Alternating A/B/B/A with CUDA events AND synchronized wall-clock timings.
    All timings exclude Colab compilation, dataset loading and model downloading."""
    results=[]
    for j,b in enumerate(BENCH_BATCHES):
        def once(m):
            torch.cuda.synchronize()
            a=torch.cuda.Event(enable_timing=True)
            z=torch.cuda.Event(enable_timing=True)
            before=time.perf_counter()
            a.record()
            with torch.autocast("cuda",dtype=amp_dtype):
                outputs=m(**b)
            z.record();z.synchronize()
            wall=(time.perf_counter()-before)*1000
            return float(a.elapsed_time(z)),float(wall)
        for k in range(4):
            once(reference);once(candidate)
        d={"reference":[],"candidate":[]}
        w={"reference":[],"candidate":[]}
        for k in range(repeats):
            order=("reference","candidate","candidate","reference") if k%2==0 else (
                "candidate","reference","reference","candidate")
            for name in order:
                gpu,cpu=once(reference if name=="reference" else candidate)
                d[name].append(gpu);w[name].append(cpu)
        ref=float(np.median(d["reference"]))
        cand=float(np.median(d["candidate"]))
        rw=float(np.median(w["reference"]))
        cw=float(np.median(w["candidate"]))
        results.append(dict(shape=j,batch=int(b["input_ids"].shape[0]),
            tokens=int(b["input_ids"].shape[1]),
            native_gpu_ms=ref,candidate_gpu_ms=cand,gpu_speedup=ref/max(cand,1e-9),
            native_wall_ms=rw,candidate_wall_ms=cw,wall_speedup=rw/max(cw,1e-9),
            native_p95_ms=float(np.percentile(d["reference"],95)),
            candidate_p95_ms=float(np.percentile(d["candidate"],95)),n=len(d["reference"])))
    return dict(median_gpu_speedup=float(np.median([r["gpu_speedup"] for r in results])),
        min_gpu_speedup=float(min(r["gpu_speedup"] for r in results)),
        median_wall_speedup=float(np.median([r["wall_speedup"] for r in results])),
        min_wall_speedup=float(min(r["wall_speedup"] for r in results)),
        per_shape=results)

def valid_quality(q):
    return (q["teacher_agreement"]>=MIN_TEACHER_AGREEMENT and
            q["gold_drop"]<=MAX_GOLD_DROP_PP/100 and q["mean_js"]<=MAX_JS)
def valid_speed(s):
    return (s["min_gpu_speedup"]>=MIN_GPU_SPEED_RATIO and
            s["min_wall_speedup"]>=MIN_WALL_SPEED_RATIO)



In [ ]:
#@title 6. SVD rank sweep on PROBE only (AUDIT untouched)
PROBE_REF=get_reference(probe_items,PROBE_DECISIONS)
def quality_gate(q):
    return (q["teacher_agreement"]>=MIN_TEACHER_AGREEMENT and
            q["gold_drop"]<=MAX_GOLD_DROP_PP/100. and q["mean_js"]<=MAX_JS)
def rank_unique_parameters(rank):
    removed=added=0
    for first,second in pairs:
        for name in ("Wqkv","Wo"):
            linear=getattr(teacher.encoder.layers[second].attn,name)
            out_dim,in_dim=linear.weight.shape
            removed+=out_dim*in_dim
            added+=int(rank)*(out_dim+in_dim)
    return native_unique-removed+added
sweep=[]
for rank in CANDIDATE_RANKS:
    v48.change_active_rank(student,rank)
    q=quality(student,probe_items,PROBE_REF)
    parameters=rank_unique_parameters(rank)
    entry=dict(rank=rank,quality=q,passed=quality_gate(q),
               expected_unique_parameters=parameters,
               expected_fp32_savings_bytes=(native_unique-parameters)*4)
    sweep.append(entry)
    (OUT/"rank_sweep.json").write_text(json.dumps(sweep,indent=2))
    print("rank",rank,"agreement",q["teacher_agreement"],
          "gold_drop",q["gold_drop"],"JS",q["mean_js"],
          "predicted_saving_MB",round(entry["expected_fp32_savings_bytes"]/1e6,2),
          "quality_pass",entry["passed"])
feasible=[r for r in sweep if r["passed"] and
          r["expected_fp32_savings_bytes"]>=MIN_SAVED_BYTES]
if feasible:
    chosen=min(feasible,key=lambda r:r["rank"])
    rank_selection="smallest_rank_passing_quality"
else:
    chosen=min(sweep,key=lambda r:r["quality"]["mean_js"])
    rank_selection="best_js_but_initially_unvalidated"
SELECTED_RANK=int(chosen["rank"])
v48.shrink_all_adapters(student,SELECTED_RANK)
unique_student=v48.unique_parameter_count(student)
assert unique_student==rank_unique_parameters(SELECTED_RANK)
print("Frozen rank:",SELECTED_RANK,"selection:",rank_selection)
print("Unique parameters:",f"{unique_student:,}",
      "saving:",f"{native_unique-unique_student:,}")


In [ ]:
#@title 7. Train only SVD residuals with frozen V4.6 teacher, choose PROBE checkpoint
adapter_params=[]
for adapter in v48.adapter_modules(student):
    adapter_params.extend((adapter.A,adapter.B))
assert len(adapter_params)==36
for p in adapter_params: p.requires_grad_(True)
assert sum(p.numel() for p in student.parameters() if p.requires_grad)==(
       sum(p.numel() for p in adapter_params))
student.eval() # dropout off but autograd ON
optimizer=torch.optim.AdamW(adapter_params,lr=TRAIN_LR,weight_decay=0.)
scaler=torch.amp.GradScaler("cuda",enabled=amp_dtype==torch.float16)
def snapshot():
    return [(a.A.detach().cpu().clone(),a.B.detach().cpu().clone())
            for a in v48.adapter_modules(student)]
def restore(snapshot_state):
    with torch.no_grad():
        for a,(A,B) in zip(v48.adapter_modules(student),snapshot_state):
            a.A.copy_(A.to(a.A.device));a.B.copy_(B.to(a.B.device))
best_state=snapshot()
best_step=0
best_quality=quality(student,probe_items,PROBE_REF)
best_pass=quality_gate(best_quality)
training_log=[]
for step in range(1,TRAIN_STEPS+1):
    selected=batch(train_items,((step-1)*TRAIN_B)%len(train_items),TRAIN_B)
    assert bool((selected["marker_pos"][selected["marker_mask"]]<TRAIN_MAX_TOKENS).all()),(
        "Option marker truncated: increase TRAIN_MAX_TOKENS")
    for key in ("input_ids","attention_mask"):
        selected[key]=selected[key][:,:TRAIN_MAX_TOKENS]
    with torch.no_grad(),torch.autocast("cuda",dtype=amp_dtype):
        teacher_logits,_=teacher(**selected)
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast("cuda",dtype=amp_dtype):
        student_logits,_=student(**selected)
        temp=2.0
        target=F.softmax(teacher_logits.float()/temp,dim=-1)
        log_probs=F.log_softmax(student_logits.float()/temp,dim=-1)
        loss=F.kl_div(log_probs,target,reduction="batchmean")*(temp**2)
    if not bool(torch.isfinite(loss)):
        raise RuntimeError(f"Nonfinite loss at step {step}")
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(adapter_params,1.)
    scaler.step(optimizer);scaler.update()
    if step%TRAIN_CHECK_EVERY==0 or step==TRAIN_STEPS:
        score=quality(student,probe_items,PROBE_REF)
        passed=quality_gate(score)
        improve=((passed and not best_pass) or
                 (passed==best_pass and score["mean_js"]<best_quality["mean_js"]))
        if improve:
            best_quality=score;best_pass=passed
            best_state=snapshot();best_step=step
        rec=dict(step=step,loss=float(loss.detach().cpu()),
                 quality=score,quality_pass=passed,selected=improve)
        training_log.append(rec)
        (OUT/"training_progress.json").write_text(json.dumps(training_log,indent=2))
        print("step",step,"loss",rec["loss"],
              "teacher_agreement",score["teacher_agreement"],
              "gold_drop",score["gold_drop"],"mean_js",score["mean_js"],
              "pass",passed,"selected",improve)
restore(best_state)
student.eval().requires_grad_(False)
print("Best PROBE checkpoint step:",best_step,"pass:",best_pass)


In [ ]:
#@title 8. Paired GPU timings then exactly ONE disjoint held-out AUDIT
frozen_probe=quality(student,probe_items,PROBE_REF)
frozen_speed=benchmark_paired(teacher,student,repeats=BENCH_REPEATS)
print("Final PROBE quality:",frozen_probe)
print("Final PROBE speed:",frozen_speed)
AUDIT_REF=get_reference(audit_items,AUDIT_DECISIONS)
assert {r["case"] for r in AUDIT_REF}.isdisjoint({r["case"] for r in PROBE_REF})
final_audit=quality(student,audit_items,AUDIT_REF)
final_speed=benchmark_paired(teacher,student,repeats=max(15,BENCH_REPEATS))
print("Held-out AUDIT:",final_audit)
print("GPU speedup ratio:",final_speed["median_gpu_speedup"])
print("GPU min ratio:",final_speed["min_gpu_speedup"])
print("Wall min ratio:",final_speed["min_wall_speedup"])


In [ ]:
#@title 9. Save complete offline model weights, test reload parity and actual bytes
from collections import OrderedDict
(DEST/"encoder").mkdir(exist_ok=True,parents=True)
(DEST/"tokenizer").mkdir(exist_ok=True)
student.encoder.config.save_pretrained(str(DEST/"encoder"))
tokenizer.save_pretrained(str(DEST/"tokenizer"))
(DEST/"modeling_laya_v48.py").write_text(V48_RUNTIME_CODE,encoding="utf-8")
cfg=dict(format="laya-v48-svd-delta-v1",
 model_type="laya-v48-svd-delta",
 head_context_budget=64,head_layers=int(source_config["head_layers"]),
 rank=SELECTED_RANK,source_model=TEACHER_REPO,source_revision=teacher_info.sha,
 shared_attention_pairs=[list(p) for p in pairs],
 initialization="truncated SVD of original follower-minus-leader weights",
 selected_training_step=best_step,train_steps_requested=TRAIN_STEPS,
 output_scope="typed-decision option scores only")
(DEST/"config.json").write_text(json.dumps(cfg,indent=2))
state=student.state_dict()
assert not [k for k in state if "._shared." in k],(
    "An aliased base weight was duplicated in checkpoint")
weights_file=DEST/"model.safetensors"
cpu_state=OrderedDict((k,v.detach().cpu().contiguous().clone()) for k,v in state.items())
save_file(cpu_state,str(weights_file),metadata={"format":"pt","variant":"v48-svd-delta"})
del cpu_state,state
gc.collect()
native_file=(parent_dir/"model.safetensors")
native_bytes=native_file.stat().st_size
student_bytes=weights_file.stat().st_size
saved_bytes=native_bytes-student_bytes
assert saved_bytes>0,("New checkpoint is not smaller",native_bytes,student_bytes)
print("Reference checkpoint:",native_bytes,"bytes")
print("V4.8 standalone checkpoint:",student_bytes,"bytes")
print("Saved:",saved_bytes,"bytes /",round(saved_bytes/1e6,2),"MB")
reloaded,reload_tokenizer=v48.load_v48_model(DEST,device="cuda")
smoke=batch(probe_items,0,BATCH)
with torch.no_grad(),torch.autocast("cuda",dtype=amp_dtype):
    original_logits,_=student(**smoke)
    reloaded_logits,_=reloaded(**smoke)
reload_max_diff=float((original_logits.float()-reloaded_logits.float()).abs().max())
reload_same=bool(torch.equal(original_logits.argmax(-1),reloaded_logits.argmax(-1)))
reload_close=bool(torch.allclose(original_logits.float(),reloaded_logits.float(),
                                 atol=2e-3,rtol=2e-3))
assert reload_same and reload_close,(reload_max_diff,reload_same,reload_close)
del reloaded
gc.collect();torch.cuda.empty_cache()
print("Offline reload parity PASSED:",reload_max_diff)


In [ ]:
#@title 10. Strict acceptance gates, complete report and model card
gates=dict(
 probe_fidelity=quality_gate(frozen_probe),
 holdout_teacher_agreement=final_audit["teacher_agreement"]>=MIN_TEACHER_AGREEMENT,
 holdout_gold_drop=final_audit["gold_drop"]<=MAX_GOLD_DROP_PP/100.,
 holdout_mean_js=final_audit["mean_js"]<=MAX_JS,
 gpu_latency_nonregression=final_speed["min_gpu_speedup"]>=MIN_GPU_SPEED_RATIO,
 wall_latency_nonregression=final_speed["min_wall_speedup"]>=MIN_WALL_SPEED_RATIO,
 saved_at_least_100MB=saved_bytes>=MIN_SAVED_BYTES,
 local_reload_parity=reload_same and reload_close)
gates["all_pass"]=all(gates.values())
report=dict(version="V4.8",architecture="FULL / LOCAL / LOCAL with SVD deltas",
 source=TEACHER_REPO,source_revision=teacher_info.sha,
 hardware=torch.cuda.get_device_name(0),torch_version=torch.__version__,
 ranks_tested=sweep,rank_selection=rank_selection,selected_rank=SELECTED_RANK,
 svd_initialization_diagnostics=svd_diagnostics,
 selected_train_step=best_step,training_log=training_log,
 native_unique_parameters=native_unique,student_unique_parameters=unique_student,
 saved_unique_parameters=native_unique-unique_student,
 baseline_checkpoint_bytes=native_bytes,student_checkpoint_bytes=student_bytes,
 saved_checkpoint_bytes=saved_bytes,
 final_probe=frozen_probe,final_probe_latency=frozen_speed,
 final_audit=final_audit,final_audit_latency=final_speed,
 reload_max_abs_diff=reload_max_diff,gates=gates)
(DEST/"v48_report.json").write_text(json.dumps(report,indent=2))
card=f"""---
library_name: pytorch
base_model: {TEACHER_REPO}
tags:
- modernbert
- low-rank
- svd
- standalone
---

# Laya V4.8 — SVD delta-shared attention

ModernBERT retains the original FULL / LOCAL(128) / LOCAL(128) layout,
separate FFNs and V4.6's 64-token marker-aware decision head.
For each local pair the second layer uses first-layer QKV/Wo plus
a trainable SVD-initialized rank-{SELECTED_RANK} residual.

- Unique parameter reduction: **{native_unique-unique_student:,}**
- Actual FP32 weight storage saved: **{saved_bytes:,} bytes**
- PROBE teacher agreement: **{frozen_probe['teacher_agreement']:.3%}**
- AUDIT teacher agreement: **{final_audit['teacher_agreement']:.3%}**
- AUDIT gold accuracy: **{final_audit['gold_accuracy']:.3%}**
- AUDIT mean JS: **{final_audit['mean_js']:.6f}**
- Median paired GPU speed ratio: **{final_speed['median_gpu_speedup']:.3f}x**
- Gates passed: **{gates['all_pass']}**

## Fully local loading

```python
from modeling_laya_v48 import load_v48_model
model,tokenizer=load_v48_model("/path/to/standalone_model",device="cuda")
answers=model.predict(tokenizer,dict(road_direction="left",speed=7),
  dict(steering=dict(type="choice",instructions="Choose road direction",
       criteria=dict(left="left",straight="straight",right="right"))))
print(answers)
```

This research experiment has no guarantee of global accuracy or speed.
Sharing full local-attention projections does not eliminate attention FLOPs,
and A/B residuals add GEMM operations. The held-out audit is limited to 200 decisions.
"""
(DEST/"README.md").write_text(card)
(DEST/"requirements.txt").write_text(
 "torch>=2.4\ntransformers==4.57.6\nsafetensors>=0.4\nhuggingface_hub>=0.25,<1\n")
def folder_bytes(path):
    return sum(p.stat().st_size for p in Path(path).rglob("*") if p.is_file())
report["model_repo_total_bytes"]=folder_bytes(DEST)
(DEST/"v48_report.json").write_text(json.dumps(report,indent=2))
print("GATES:",json.dumps(gates,indent=2))
print("FULL MODEL:",weights_file)
print("MODEL SIZE:",folder_bytes(DEST),"bytes")
if not gates["all_pass"]:
    print("NOT VALIDATED: checkpoint saved locally, Hugging Face upload will be blocked")


In [ ]:
#@title 11. Upload separate private HF model ONLY if EVERY gate passes
if not gates["all_pass"]:
    print("Upload BLOCKED: one or more quality/speed/size/reload gates failed.")
elif PUBLISH_TO_HF:
    notebook_login()
    publisher=HfApi()
    account=publisher.whoami()["name"]
    assert account=="vtava",f"Expected vtava HF account; got {account!r}"
    assert STUDENT_REPO=="vtava/Laya-v48-svd-delta-shared"
    if publisher.repo_exists(STUDENT_REPO,repo_type="model"):
        existing=set(publisher.list_repo_files(STUDENT_REPO,repo_type="model"))
        if existing-{".gitattributes",".gitignore"} and not ALLOW_UPDATE_EXISTING:
            raise RuntimeError("V4.8 HF repository already contains files. "
                               "Set ALLOW_UPDATE_EXISTING=True only to replace deliberately.")
    else:
        created=publisher.create_repo(repo_id=STUDENT_REPO,repo_type="model",
                                      private=PRIVATE_REPO,exist_ok=False)
        print("Created:",str(created))
    publisher.upload_large_folder(
        repo_id=STUDENT_REPO,repo_type="model",folder_path=str(DEST))
    remote=set(publisher.list_repo_files(STUDENT_REPO,repo_type="model"))
    required={"model.safetensors","modeling_laya_v48.py","config.json",
              "encoder/config.json","v48_report.json","README.md"}
    assert required.issubset(remote),f"Remote upload missing: {required-remote}"
    print("FULL MODEL VERIFIED:",f"https://huggingface.co/{STUDENT_REPO}")
else:
    print("Publishing disabled; local model and reports:",DEST)
print("Original V4.6 reference unchanged:",TEACHER_REPO)
